In [ ]:
# ============================================================
# CELL 6 — FIRST CELL OF A **BRAND NEW** COLAB NOTEBOOK.
#
# Before running: Runtime > Change runtime type > T4 GPU.
# Also upload handoff.json (folder icon on the left, the upload arrow at the top).
#
# This installs NeMo into a clean environment. Nothing else is installed, so
# there is no whisper/pyannote/numba conflict to resolve.
# Takes 5-10 minutes and prints many warnings. Warnings are fine.
# When it finishes: Runtime > Restart session, then run CELL 7.
# ============================================================

!apt-get -qq update > /dev/null && apt-get -qq install -y libsndfile1 ffmpeg > /dev/null
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr]"

print()
print("=" * 60)
print("NeMo installed. Do NOT upgrade numba - NeMo pins what it needs.")
print("NEXT: Runtime > Restart session, then run CELL 7.")
print("=" * 60)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 7 — SECOND CELL OF THE NEW NOTEBOOK. Run after the restart.
#
# Needs /content/handoff.json uploaded (folder icon > upload arrow).
#
# Sortformer and Parakeet run one at a time, and each is freed before the next
# loads, so peak memory is one model rather than two.
# ============================================================

import os, sys, json, time, gc, threading, subprocess
import torch, psutil, numpy as np, soundfile as sf, pandas as pd

# --- repo, for the reference scripts and the project's own metric code -----
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

# Pure-python: the app imports live inside main(), so no database is touched.
from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, audio_duration, SCRIPTS_MD,
)
scripts = parse_scripts(SCRIPTS_MD)

with open("/content/handoff.json") as f:
    HANDOFF = json.load(f)
print(f"handoff: {len(HANDOFF['scripts'])} scripts, "
      f"measured on {HANDOFF['hardware'].get('gpu')}")

import platform
HARDWARE = {
    "platform": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_cores_physical": psutil.cpu_count(logical=False),
    "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                   if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
}
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU."
print(json.dumps(HARDWARE, indent=2))


class Measured:
    """Records VRAM, RAM, CPU and wall time for a block."""

    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def score(n, segments):
    """Word and speaker accuracy against the scripted reference."""
    ref_words, ref_spk = [], []
    for speaker, text in scripts[n]:
        w = normalise(text)
        ref_words.extend(w)
        ref_spk.extend([speaker] * len(w))
    hyp_words, hyp_spk = [], []
    for seg in segments:
        w = normalise(seg["text"])
        hyp_words.extend(w)
        hyp_spk.extend([seg["speaker_role"]] * len(w))
    wacc = max(0.0, 1 - word_error_rate(
        strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    return wacc, (correct / total * 100) if total else 0.0


_SF = {}

def mono16k(wav):
    """16 kHz mono copy under /content; the evidence files are left alone."""
    if wav in _SF:
        return _SF[wav]
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    _SF[wav] = out
    return out


AUDIO_DIR = HANDOFF["audio_dir"]
rows = []


def record(label, n, wacc, spk, segs, stage_stats):
    """One row. Stage stats are summed: these pipelines run in sequence."""
    d = HANDOFF["scripts"][str(n)]["audio_s"]
    infer = round(sum(s["wall_s"] for s in stage_stats), 2)
    rows.append({
        "run": label, "audio_set": os.path.basename(AUDIO_DIR), "script": n,
        "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
        "segments": segs, "ref_turns": len(scripts[n]), "audio_s": round(d, 1),
        "infer_s": infer, "realtime_x": round(infer / d, 2) if d else None,
        "cpu_s": round(sum(s["cpu_s"] for s in stage_stats), 2),
        "vram_peak_gb": round(max(s["vram_peak_gb"] or 0 for s in stage_stats), 2),
        "ram_peak_gb": round(max(s["ram_peak_gb"] for s in stage_stats), 2),
    })
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  {infer}s")


# ===================== Run 2: Whisper medium + Sortformer =================
# Whisper's words come from the handoff file; Sortformer replaces pyannote,
# so any difference against Run 1 is the diarizer's doing and nothing else.
print("\n=== 2. medium + sortformer ===")
from nemo.collections.asr.models import SortformerEncLabelModel

with Measured("load") as load_sf:
    sortformer = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
    sortformer.eval()
    if torch.cuda.is_available():
        sortformer = sortformer.cuda()
print(f"  load {load_sf.stats['wall_s']}s, VRAM {load_sf.stats['vram_peak_gb']}GB")

first = True
for n in sorted(scripts):
    d = HANDOFF["scripts"][str(n)]
    wav = mono16k(f"{AUDIO_DIR}/consult_{n}.wav")

    with Measured(f"sf{n}") as m:
        pred = sortformer.diarize(audio=[wav], batch_size=1)

    segs_raw = pred[0] if isinstance(pred, (list, tuple)) and pred else pred
    spans = []
    for s in segs_raw:
        if isinstance(s, str):
            p = s.replace(",", " ").split()
            spans.append((float(p[0]), float(p[1]), str(p[2])))
        elif isinstance(s, (list, tuple)) and len(s) >= 3:
            spans.append((float(s[0]), float(s[1]), str(s[2])))
        else:
            raise RuntimeError(f"unexpected Sortformer segment: {type(s)} {s!r}")
    if first:
        print(f"  [sortformer: {len(spans)} spans, first={spans[0] if spans else None}]")
        first = False

    segments = words_to_turns(d["whisper_medium_words"], spans)
    wacc, spk = score(n, segments)
    record("2. medium + sortformer", n, wacc, spk, len(segments),
           [d["whisper_medium_stats"], m.stats])

del sortformer
gc.collect()
torch.cuda.empty_cache()


# ===================== Run 3: Parakeet + pyannote =========================
# Parakeet replaces Whisper; the pyannote spans come from the handoff file,
# so this isolates the ASR.
print("\n=== 3. parakeet + pyannote ===")
import nemo.collections.asr as nemo_asr

with Measured("load") as load_pk:
    parakeet = nemo_asr.models.ASRModel.from_pretrained(
        model_name="nvidia/parakeet-tdt-0.6b-v2")
    parakeet.eval()
print(f"  load {load_pk.stats['wall_s']}s, VRAM {load_pk.stats['vram_peak_gb']}GB")

first = True
for n in sorted(scripts):
    d = HANDOFF["scripts"][str(n)]
    wav = mono16k(f"{AUDIO_DIR}/consult_{n}.wav")

    with Measured(f"pk{n}") as m:
        out = parakeet.transcribe([wav], timestamps=True)

    # Parakeet's tokens carry no leading space; Whisper's do, and words_to_turns
    # concatenates directly. Without the space the words glue together and the
    # word-accuracy figure becomes meaningless.
    words = [{"start": float(s["start"]), "end": float(s["end"]),
              "word": " " + str(s["word"]).strip()}
             for s in out[0].timestamp["word"]]
    if first:
        print(f"  [parakeet: {len(words)} words, first={words[0] if words else None}]")
        first = False

    spans = [tuple(s) for s in d["pyannote_spans"]]
    segments = words_to_turns(words, spans)
    wacc, spk = score(n, segments)
    record("3. parakeet + pyannote", n, wacc, spk, len(segments),
           [m.stats, d["pyannote_stats"]])


# ===================== export ============================================
detail = pd.DataFrame(rows)
summary = (detail.groupby("run")
           .agg(word_acc_mean=("word_acc", "mean"),
                speaker_acc_mean=("speaker_acc", "mean"),
                infer_s_mean=("infer_s", "mean"),
                realtime_x_mean=("realtime_x", "mean"),
                cpu_s_mean=("cpu_s", "mean"),
                vram_peak_gb=("vram_peak_gb", "max"),
                ram_peak_gb=("ram_peak_gb", "max"))
           .round(2).reset_index())
summary["load_s"] = [load_sf.stats["wall_s"], load_pk.stats["wall_s"]]
summary["load_vram_gb"] = [load_sf.stats["vram_peak_gb"], load_pk.stats["vram_peak_gb"]]

print()
print("=" * 70)
display(summary)

detail.to_csv("/content/nemo_detail.csv", index=False)
summary.to_csv("/content/nemo_comparison.csv", index=False)
with open("/content/nemo_hardware.json", "w") as f:
    json.dump(HARDWARE, f, indent=2)
print("\nWritten: nemo_detail.csv, nemo_comparison.csv, nemo_hardware.json")
print("Download all three and send them to me.")

handoff: 4 scripts, measured on Tesla T4
{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_cores_physical": 1,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}

=== 2. medium + sortformer ===


diar_sortformer_4spk-v1.nemo: reconstructing file:   0%|          |  0.00B /  493MB            

diar_sortformer_4spk-v1.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-08-19 13:03:10 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-19 13:03:10 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-19 13:03:13 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
  load 11.48s, VRAM 1.01GB
[NeMo I 2026-08-19 13:03:13 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


...

[NeMo W 2026-08-19 13:04:01 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-08-19 13:04:01 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_

[NeMo I 2026-08-19 13:04:07 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:04:07 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:04:07 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:04:17 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/ae9ad07059c7c739ffaf932226a8fe64ae2620b0/parakeet-tdt-0.6b-v2.nemo.
  load 57.23s, VRAM 4.99GB
[NeMo I 2026-08-19 13:04:18 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                  

[NeMo W 2026-08-19 13:04:18 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 13:04:18 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.56s/it]


  [parakeet: 215 words, first={'start': 0.96, 'end': 1.44, 'word': ' Good'}]
  script 1: word 90.6%  speaker 100.0%  7.45s
[NeMo I 2026-08-19 13:04:20 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']


[NeMo W 2026-08-19 13:04:20 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 13:04:20 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s]

In [ ]:
# ============================================================
# CELL 8 — REPLACES CELL 7. Run this instead.
#
# Before running: Runtime > Restart session (to clear the crashed state).
# handoff.json must still be uploaded — check the folder panel on the left.
#
# Two changes from Cell 7:
#   1. Every script's result is written to /content/nemo_detail.csv the moment
#      it is computed, and already-finished work is skipped on a re-run. A crash
#      now costs one script, not the whole notebook.
#   2. Parakeet transcribes with num_workers=0. NeMo's dataloader forks worker
#      processes by default and each fork copies the parent's memory, which is
#      what killed the last run on a 13.6 GB machine.
# ============================================================

import os, sys, json, time, gc, threading, subprocess
import torch, psutil, soundfile as sf, pandas as pd

if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, SCRIPTS_MD,
)
scripts = parse_scripts(SCRIPTS_MD)

with open("/content/handoff.json") as f:
    HANDOFF = json.load(f)

import platform
HARDWARE = {
    "platform": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_cores_physical": psutil.cpu_count(logical=False),
    "cpu_cores_logical": psutil.cpu_count(logical=True),
    "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                   if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
}
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU."
with open("/content/nemo_hardware.json", "w") as f:
    json.dump(HARDWARE, f, indent=2)


class Measured:
    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def score(n, segments):
    ref_words, ref_spk = [], []
    for speaker, text in scripts[n]:
        w = normalise(text)
        ref_words.extend(w)
        ref_spk.extend([speaker] * len(w))
    hyp_words, hyp_spk = [], []
    for seg in segments:
        w = normalise(seg["text"])
        hyp_words.extend(w)
        hyp_spk.extend([seg["speaker_role"]] * len(w))
    wacc = max(0.0, 1 - word_error_rate(
        strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    return wacc, (correct / total * 100) if total else 0.0


_SF = {}

def mono16k(wav):
    if wav in _SF:
        return _SF[wav]
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    _SF[wav] = out
    return out


# --------------------------------------------------------------- persistence
DETAIL = "/content/nemo_detail.csv"
AUDIO_DIR = HANDOFF["audio_dir"]

def done_already():
    if not os.path.exists(DETAIL):
        return set()
    d = pd.read_csv(DETAIL)
    return set(zip(d.run, d.script))

DONE = done_already()
if DONE:
    print(f"resuming — already have: {sorted(DONE)}")


def record(label, n, wacc, spk, segs, stage_stats, load_stats):
    """Append one row immediately. Stage costs are combined the way the pipeline
    actually runs: time and CPU add up because the stages run in sequence, but
    peak memory is the larger stage, not the sum, because they are not resident
    together."""
    d = HANDOFF["scripts"][str(n)]["audio_s"]
    infer = round(sum(s["wall_s"] for s in stage_stats), 2)
    row = {
        "run": label, "audio_set": os.path.basename(AUDIO_DIR), "script": n,
        "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
        "segments": segs, "ref_turns": len(scripts[n]), "audio_s": round(d, 1),
        "infer_s": infer, "realtime_x": round(infer / d, 2) if d else None,
        "cpu_s": round(sum(s["cpu_s"] for s in stage_stats), 2),
        "vram_peak_gb": round(max(s["vram_peak_gb"] or 0 for s in stage_stats), 2),
        "ram_peak_gb": round(max(s["ram_peak_gb"] for s in stage_stats), 2),
        "load_s": load_stats["wall_s"],
        "load_vram_gb": load_stats["vram_peak_gb"],
    }
    pd.DataFrame([row]).to_csv(DETAIL, mode="a", header=not os.path.exists(DETAIL),
                               index=False)
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  {infer}s  [saved]")


# ================= Run 2: Whisper medium words + Sortformer ===============
LABEL2 = "2. medium + sortformer"
todo2 = [n for n in sorted(scripts) if (LABEL2, n) not in DONE]

if todo2:
    print(f"\n=== {LABEL2} ===")
    from nemo.collections.asr.models import SortformerEncLabelModel

    with Measured("load") as load_sf:
        sortformer = SortformerEncLabelModel.from_pretrained(
            "nvidia/diar_sortformer_4spk-v1")
        sortformer.eval()
        if torch.cuda.is_available():
            sortformer = sortformer.cuda()
    print(f"  load {load_sf.stats['wall_s']}s, VRAM {load_sf.stats['vram_peak_gb']}GB")

    for n in todo2:
        d = HANDOFF["scripts"][str(n)]
        wav = mono16k(f"{AUDIO_DIR}/consult_{n}.wav")

        with Measured(f"sf{n}") as m:
            pred = sortformer.diarize(audio=[wav], batch_size=1)

        raw = pred[0] if isinstance(pred, (list, tuple)) and pred else pred
        spans = []
        for s in raw:
            if isinstance(s, str):
                p = s.replace(",", " ").split()
                spans.append((float(p[0]), float(p[1]), str(p[2])))
            elif isinstance(s, (list, tuple)) and len(s) >= 3:
                spans.append((float(s[0]), float(s[1]), str(s[2])))
            else:
                raise RuntimeError(f"unexpected Sortformer segment: {type(s)} {s!r}")

        segments = words_to_turns(d["whisper_medium_words"], spans)
        wacc, spk = score(n, segments)
        record(LABEL2, n, wacc, spk, len(segments),
               [d["whisper_medium_stats"], m.stats], load_sf.stats)

    del sortformer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"\n=== {LABEL2} — already complete, skipping ===")


# ================= Run 3: Parakeet + pyannote spans =======================
LABEL3 = "3. parakeet + pyannote"
todo3 = [n for n in sorted(scripts) if (LABEL3, n) not in DONE]

if todo3:
    print(f"\n=== {LABEL3} ===")
    import nemo.collections.asr as nemo_asr

    with Measured("load") as load_pk:
        parakeet = nemo_asr.models.ASRModel.from_pretrained(
            model_name="nvidia/parakeet-tdt-0.6b-v2")
        parakeet.eval()
    print(f"  load {load_pk.stats['wall_s']}s, VRAM {load_pk.stats['vram_peak_gb']}GB")

    for n in todo3:
        d = HANDOFF["scripts"][str(n)]
        wav = mono16k(f"{AUDIO_DIR}/consult_{n}.wav")

        with Measured(f"pk{n}") as m:
            # num_workers=0 keeps the dataloader in this process. The default
            # forks workers, and on this machine the forked copies exhaust RAM.
            out = parakeet.transcribe([wav], timestamps=True,
                                      batch_size=1, num_workers=0)

        # Parakeet's tokens carry no leading space; Whisper's do, and
        # words_to_turns concatenates directly. Without the space the words glue
        # together and the word-accuracy figure becomes meaningless.
        words = [{"start": float(s["start"]), "end": float(s["end"]),
                  "word": " " + str(s["word"]).strip()}
                 for s in out[0].timestamp["word"]]

        spans = [tuple(s) for s in d["pyannote_spans"]]
        segments = words_to_turns(words, spans)
        wacc, spk = score(n, segments)
        record(LABEL3, n, wacc, spk, len(segments),
               [m.stats, d["pyannote_stats"]], load_pk.stats)

        gc.collect()
        torch.cuda.empty_cache()
else:
    print(f"\n=== {LABEL3} — already complete, skipping ===")


# ------------------------------------------------------------------ summary
detail = pd.read_csv(DETAIL)
summary = (detail.groupby("run")
           .agg(word_acc_mean=("word_acc", "mean"),
                speaker_acc_mean=("speaker_acc", "mean"),
                infer_s_mean=("infer_s", "mean"),
                realtime_x_mean=("realtime_x", "mean"),
                cpu_s_mean=("cpu_s", "mean"),
                vram_peak_gb=("vram_peak_gb", "max"),
                ram_peak_gb=("ram_peak_gb", "max"),
                load_s=("load_s", "max"),
                load_vram_gb=("load_vram_gb", "max"))
           .round(2).reset_index())
summary.to_csv("/content/nemo_comparison.csv", index=False)

print()
print("=" * 70)
display(summary)
missing = [(l, n) for l in (LABEL2, LABEL3) for n in sorted(scripts)
           if (l, n) not in set(zip(detail.run, detail.script))]
if missing:
    print(f"\nSTILL MISSING: {missing} — re-run this cell and it will pick up "
          "only those.")
else:
    print("\nAll runs complete. Download nemo_detail.csv, nemo_comparison.csv "
          "and nemo_hardware.json.")



=== 2. medium + sortformer ===


[NeMo W 2026-08-19 13:11:00 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-19 13:11:00 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-19 13:11:02 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
  load 4.33s, VRAM 1.01GB
[NeMo I 2026-08-19 13:11:02 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-08-19 13:11:02 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: session_len_sec,soft_label_thres,num_spks
Diarizing: 1it [00:01,  1.51s/it]


  script 1: word 90.6%  speaker 100.0%  20.12s  [saved]
[NeMo I 2026-08-19 13:11:04 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


...

[NeMo W 2026-08-19 13:11:29 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-08-19 13:11:29 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_

[NeMo I 2026-08-19 13:11:34 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:11:34 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:11:34 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:11:42 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/ae9ad07059c7c739ffaf932226a8fe64ae2620b0/parakeet-tdt-0.6b-v2.nemo.
  load 34.91s, VRAM 4.99GB
[NeMo I 2026-08-19 13:11:43 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                  

[NeMo W 2026-08-19 13:11:43 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 13:11:43 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.30s/it]


  script 1: word 90.6%  speaker 100.0%  7.12s  [saved]
[NeMo I 2026-08-19 13:11:45 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']


[NeMo W 2026-08-19 13:11:45 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 13:11:45 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s]

In [ ]:
# ============================================================
# CELL 9 — FINISH PARAKEET. Run after: Runtime > Restart session.
#
# Sortformer is done and saved; this touches only the missing Parakeet scripts.
#
# Why this is different: the crash happened on the SECOND transcribe() call,
# both times, regardless of which audio was next. NeMo rebuilds its dataloader
# per call and that is what dies. So this makes exactly ONE call, passing every
# remaining file at once.
#
# Cost of that: the ASR stage is timed for the batch as a whole, not per script.
# Accuracy is still exact per script. The rows are marked so the report can say
# so rather than implying four independent measurements.
# ============================================================

import os, sys, json, time, gc, threading, subprocess
import torch, psutil, soundfile as sf, pandas as pd

os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, SCRIPTS_MD,
)
scripts = parse_scripts(SCRIPTS_MD)

with open("/content/handoff.json") as f:
    HANDOFF = json.load(f)
AUDIO_DIR = HANDOFF["audio_dir"]
DETAIL = "/content/nemo_detail.csv"
LABEL3 = "3. parakeet + pyannote"

done = pd.read_csv(DETAIL)
have = set(zip(done.run, done.script))
todo = [n for n in sorted(scripts) if (LABEL3, n) not in have]
print(f"already saved: {sorted(have)}")
print(f"still to do:   {todo}")
if not todo:
    raise SystemExit("Parakeet already complete — skip to the summary cell.")


class Measured:
    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False


def assign_roles(segments):
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def score(n, segments):
    ref_words, ref_spk = [], []
    for speaker, text in scripts[n]:
        w = normalise(text)
        ref_words.extend(w)
        ref_spk.extend([speaker] * len(w))
    hyp_words, hyp_spk = [], []
    for seg in segments:
        w = normalise(seg["text"])
        hyp_words.extend(w)
        hyp_spk.extend([seg["speaker_role"]] * len(w))
    wacc = max(0.0, 1 - word_error_rate(
        strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    return wacc, (correct / total * 100) if total else 0.0


def mono16k(wav):
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    return out


# ---------------------------------------------------------------- one call
import nemo.collections.asr as nemo_asr

with Measured("load") as load_pk:
    parakeet = nemo_asr.models.ASRModel.from_pretrained(
        model_name="nvidia/parakeet-tdt-0.6b-v2")
    parakeet.eval()
print(f"  load {load_pk.stats['wall_s']}s, VRAM {load_pk.stats['vram_peak_gb']}GB")

wavs = [mono16k(f"{AUDIO_DIR}/consult_{n}.wav") for n in todo]
print(f"  transcribing {len(wavs)} files in a single call ...")

with Measured("asr_batch") as m:
    out = parakeet.transcribe(wavs, timestamps=True, batch_size=1, num_workers=0)

print(f"  batch ASR: {m.stats['wall_s']}s for {len(wavs)} files, "
      f"VRAM {m.stats['vram_peak_gb']}GB")

per_file_asr_s = round(m.stats["wall_s"] / len(wavs), 2)
per_file_cpu_s = round(m.stats["cpu_s"] / len(wavs), 2)

rows = []
for n, hyp in zip(todo, out):
    d = HANDOFF["scripts"][str(n)]
    # Parakeet's tokens carry no leading space; Whisper's do, and words_to_turns
    # concatenates directly. Without the space the words glue together and the
    # word-accuracy figure becomes meaningless.
    words = [{"start": float(s["start"]), "end": float(s["end"]),
              "word": " " + str(s["word"]).strip()}
             for s in hyp.timestamp["word"]]
    spans = [tuple(s) for s in d["pyannote_spans"]]
    segments = words_to_turns(words, spans)
    wacc, spk = score(n, segments)

    infer = round(per_file_asr_s + d["pyannote_stats"]["wall_s"], 2)
    rows.append({
        "run": LABEL3, "audio_set": os.path.basename(AUDIO_DIR), "script": n,
        "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
        "segments": len(segments), "ref_turns": len(scripts[n]),
        "audio_s": round(d["audio_s"], 1),
        "infer_s": infer,
        "realtime_x": round(infer / d["audio_s"], 2) if d["audio_s"] else None,
        "cpu_s": round(per_file_cpu_s + d["pyannote_stats"]["cpu_s"], 2),
        "vram_peak_gb": round(max(m.stats["vram_peak_gb"] or 0,
                                  d["pyannote_stats"]["vram_peak_gb"] or 0), 2),
        "ram_peak_gb": round(max(m.stats["ram_peak_gb"],
                                 d["pyannote_stats"]["ram_peak_gb"]), 2),
        "load_s": load_pk.stats["wall_s"],
        "load_vram_gb": load_pk.stats["vram_peak_gb"],
        # Flag: the ASR stage was timed across the batch and divided, because
        # calling transcribe() once per file crashes the runtime. Accuracy is
        # exact per script; ASR timing is a batch mean.
        "asr_timing": "batch_mean",
    })
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%")

# Rewrite rather than append: these rows carry an extra "asr_timing" column,
# and appending without a header to a file with different columns would shift
# every value one place to the left in the saved file.
detail = pd.concat([pd.read_csv(DETAIL), pd.DataFrame(rows)], ignore_index=True)
detail.to_csv(DETAIL, index=False)
print(f"\nsaved {len(detail)} rows to {DETAIL}")

# ------------------------------------------------------------------ summary
summary = (detail.groupby("run")
           .agg(word_acc_mean=("word_acc", "mean"),
                speaker_acc_mean=("speaker_acc", "mean"),
                infer_s_mean=("infer_s", "mean"),
                realtime_x_mean=("realtime_x", "mean"),
                cpu_s_mean=("cpu_s", "mean"),
                vram_peak_gb=("vram_peak_gb", "max"),
                ram_peak_gb=("ram_peak_gb", "max"),
                load_s=("load_s", "max"),
                load_vram_gb=("load_vram_gb", "max"))
           .round(2).reset_index())
summary.to_csv("/content/nemo_comparison.csv", index=False)
display(summary)
print("\nDownload nemo_detail.csv, nemo_comparison.csv and nemo_hardware.json.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/repo'

In [ ]:
import os, subprocess

if not os.path.isdir("/content/repo"):
    print("cloning repo ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)

for f in ("/content/handoff.json", "/content/nemo_detail.csv",
          "/content/nemo_hardware.json"):
    print(f"{'OK      ' if os.path.exists(f) else 'MISSING '} {f}")

cloning repo ...
MISSING  /content/handoff.json
MISSING  /content/nemo_detail.csv
MISSING  /content/nemo_hardware.json


In [ ]:
try:
    import nemo.collections.asr  # noqa
    print("NeMo present — go straight to step 3")
except ImportError:
    print("NeMo gone — run PASTE_6 again, restart, then step 3")

NeMo gone — run PASTE_6 again, restart, then step 3


In [ ]:
# ============================================================
# CELL 10 — SORTFORMER AND PARAKEET IN ONE PASS. No restart in between.
#
# Before running: upload handoff.json (folder icon on the left, upload arrow).
#
# Sortformer first, saved to disk as each script finishes. Then Parakeet, in a
# SINGLE transcribe() call covering all four files — calling it once per file
# crashed the runtime twice, at the second call both times.
#
# If this crashes at Parakeet, the Sortformer results are already on disk and
# the summary at the bottom can be produced from them alone.
# ============================================================

import os, sys, json, time, gc, threading, subprocess, platform
import torch, psutil, soundfile as sf, pandas as pd

if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

assert os.path.exists("/content/handoff.json"), (
    "handoff.json is not uploaded. Folder icon on the left > upload arrow.")

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, SCRIPTS_MD,
)
scripts = parse_scripts(SCRIPTS_MD)

with open("/content/handoff.json") as f:
    HANDOFF = json.load(f)
AUDIO_DIR = HANDOFF["audio_dir"]
DETAIL = "/content/nemo_detail.csv"

HARDWARE = {
    "platform": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_cores_physical": psutil.cpu_count(logical=False),
    "cpu_cores_logical": psutil.cpu_count(logical=True),
    "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                   if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda,
}
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU."
with open("/content/nemo_hardware.json", "w") as f:
    json.dump(HARDWARE, f, indent=2)
print(json.dumps(HARDWARE, indent=2))


class Measured:
    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def score(n, segments):
    ref_words, ref_spk = [], []
    for speaker, text in scripts[n]:
        w = normalise(text)
        ref_words.extend(w)
        ref_spk.extend([speaker] * len(w))
    hyp_words, hyp_spk = [], []
    for seg in segments:
        w = normalise(seg["text"])
        hyp_words.extend(w)
        hyp_spk.extend([seg["speaker_role"]] * len(w))
    wacc = max(0.0, 1 - word_error_rate(
        strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
    correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
    return wacc, (correct / total * 100) if total else 0.0


_SF = {}

def mono16k(wav):
    if wav in _SF:
        return _SF[wav]
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    _SF[wav] = out
    return out


COLUMNS = ["run", "audio_set", "script", "word_acc", "speaker_acc", "segments",
           "ref_turns", "audio_s", "infer_s", "realtime_x", "cpu_s",
           "vram_peak_gb", "ram_peak_gb", "load_s", "load_vram_gb", "asr_timing"]

def append_rows(rows):
    df = pd.DataFrame(rows).reindex(columns=COLUMNS)
    df.to_csv(DETAIL, mode="a", header=not os.path.exists(DETAIL), index=False)


def make_row(label, n, wacc, spk, segs, infer_s, cpu_s, vram, ram,
             load_stats, asr_timing):
    d = HANDOFF["scripts"][str(n)]["audio_s"]
    return {
        "run": label, "audio_set": os.path.basename(AUDIO_DIR), "script": n,
        "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
        "segments": segs, "ref_turns": len(scripts[n]), "audio_s": round(d, 1),
        "infer_s": round(infer_s, 2),
        "realtime_x": round(infer_s / d, 2) if d else None,
        "cpu_s": round(cpu_s, 2), "vram_peak_gb": round(vram, 2),
        "ram_peak_gb": round(ram, 2),
        "load_s": load_stats["wall_s"], "load_vram_gb": load_stats["vram_peak_gb"],
        "asr_timing": asr_timing,
    }


# ================= Run 2: Whisper medium words + Sortformer ===============
LABEL2 = "2. medium + sortformer"
print(f"\n=== {LABEL2} ===")
from nemo.collections.asr.models import SortformerEncLabelModel

with Measured("load") as load_sf:
    sortformer = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
    sortformer.eval()
    if torch.cuda.is_available():
        sortformer = sortformer.cuda()
print(f"  load {load_sf.stats['wall_s']}s, VRAM {load_sf.stats['vram_peak_gb']}GB")

for n in sorted(scripts):
    d = HANDOFF["scripts"][str(n)]
    with Measured(f"sf{n}") as m:
        pred = sortformer.diarize(audio=[mono16k(f"{AUDIO_DIR}/consult_{n}.wav")],
                                  batch_size=1)

    raw = pred[0] if isinstance(pred, (list, tuple)) and pred else pred
    spans = []
    for s in raw:
        if isinstance(s, str):
            p = s.replace(",", " ").split()
            spans.append((float(p[0]), float(p[1]), str(p[2])))
        elif isinstance(s, (list, tuple)) and len(s) >= 3:
            spans.append((float(s[0]), float(s[1]), str(s[2])))
        else:
            raise RuntimeError(f"unexpected Sortformer segment: {type(s)} {s!r}")

    segments = words_to_turns(d["whisper_medium_words"], spans)
    wacc, spk = score(n, segments)
    w = d["whisper_medium_stats"]
    append_rows([make_row(
        LABEL2, n, wacc, spk, len(segments),
        w["wall_s"] + m.stats["wall_s"],          # stages run in sequence
        w["cpu_s"] + m.stats["cpu_s"],
        max(w["vram_peak_gb"] or 0, m.stats["vram_peak_gb"] or 0),  # not co-resident
        max(w["ram_peak_gb"], m.stats["ram_peak_gb"]),
        load_sf.stats, "per_file")])
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  [saved]")

del sortformer
gc.collect()
torch.cuda.empty_cache()
print("\n>>> Sortformer complete and saved. Download nemo_detail.csv NOW if you")
print(">>> want it safe before Parakeet runs.")


# ================= Run 3: Parakeet + pyannote spans =======================
LABEL3 = "3. parakeet + pyannote"
print(f"\n=== {LABEL3} ===")
import nemo.collections.asr as nemo_asr

with Measured("load") as load_pk:
    parakeet = nemo_asr.models.ASRModel.from_pretrained(
        model_name="nvidia/parakeet-tdt-0.6b-v2")
    parakeet.eval()
print(f"  load {load_pk.stats['wall_s']}s, VRAM {load_pk.stats['vram_peak_gb']}GB")

todo = sorted(scripts)
wavs = [mono16k(f"{AUDIO_DIR}/consult_{n}.wav") for n in todo]
print(f"  transcribing {len(wavs)} files in a single call ...")

with Measured("asr_batch") as m:
    out = parakeet.transcribe(wavs, timestamps=True, batch_size=1, num_workers=0)

print(f"  batch ASR: {m.stats['wall_s']}s for {len(wavs)} files, "
      f"VRAM {m.stats['vram_peak_gb']}GB")

per_asr_s = m.stats["wall_s"] / len(wavs)
per_asr_cpu = m.stats["cpu_s"] / len(wavs)

rows = []
for n, hyp in zip(todo, out):
    d = HANDOFF["scripts"][str(n)]
    # Parakeet's tokens carry no leading space; Whisper's do, and words_to_turns
    # concatenates directly. Without the space the words glue together and the
    # word-accuracy figure becomes meaningless.
    words = [{"start": float(s["start"]), "end": float(s["end"]),
              "word": " " + str(s["word"]).strip()}
             for s in hyp.timestamp["word"]]
    segments = words_to_turns(words, [tuple(s) for s in d["pyannote_spans"]])
    wacc, spk = score(n, segments)
    p = d["pyannote_stats"]
    rows.append(make_row(
        LABEL3, n, wacc, spk, len(segments),
        per_asr_s + p["wall_s"], per_asr_cpu + p["cpu_s"],
        max(m.stats["vram_peak_gb"] or 0, p["vram_peak_gb"] or 0),
        max(m.stats["ram_peak_gb"], p["ram_peak_gb"]),
        load_pk.stats, "batch_mean"))
    print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%")

append_rows(rows)


# ------------------------------------------------------------------ summary
detail = pd.read_csv(DETAIL)
summary = (detail.groupby("run")
           .agg(word_acc_mean=("word_acc", "mean"),
                speaker_acc_mean=("speaker_acc", "mean"),
                infer_s_mean=("infer_s", "mean"),
                realtime_x_mean=("realtime_x", "mean"),
                cpu_s_mean=("cpu_s", "mean"),
                vram_peak_gb=("vram_peak_gb", "max"),
                ram_peak_gb=("ram_peak_gb", "max"),
                load_s=("load_s", "max"),
                load_vram_gb=("load_vram_gb", "max"))
           .round(2).reset_index())
summary.to_csv("/content/nemo_comparison.csv", index=False)
print()
display(summary)
print("\nDownload nemo_detail.csv, nemo_comparison.csv and nemo_hardware.json NOW.")

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}

=== 2. medium + sortformer ===


diar_sortformer_4spk-v1.nemo: reconstructing file:   0%|          |  0.00B /  493MB            

diar_sortformer_4spk-v1.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-08-19 13:24:52 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-19 13:24:52 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-19 13:24:54 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
  load 10.64s, VRAM 1.01GB
[NeMo I 2026-08-19 13:24:55 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


...

[NeMo I 2026-08-19 13:25:38 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:25:38 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:25:38 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 13:25:41 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/ae9ad07059c7c739ffaf932226a8fe64ae2620b0/parakeet-tdt-0.6b-v2.nemo.
  load 39.89s, VRAM 4.99GB
  transcribing 4 files in a single call ...
[NeMo I 2026-08-19 13:25:42 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture 

[NeMo W 2026-08-19 13:25:42 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 13:25:42 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 4it [00:04,  1.07s/it]


  batch ASR: 4.35s for 4 files, VRAM 3.54GB
  script 1: word 90.6%  speaker 100.0%
  script 2: word 94.8%  speaker 12.9%
  script 3: word 89.0%  speaker 98.1%
  script 4: word 86.7%  speaker 99.7%



,run,word_acc_mean,speaker_acc_mean,infer_s_mean,realtime_x_mean,cpu_s_mean,vram_peak_gb,ram_peak_gb,load_s,load_vram_gb
0,2. medium + sortformer,89.02,99.85,15.99,0.17,14.10,3.50,3.62,10.64,1.01
1,3. parakeet + pyannote,90.28,77.68,6.03,0.06,5.76,4.77,4.04,39.89,4.99



Download nemo_detail.csv, nemo_comparison.csv and nemo_hardware.json NOW.
